# Universal Semantic Manifold — v2

**Part 2** of the USM research. This notebook runs the enhanced pipeline with:

1. **Learnable curvature** — the Poincaré ball curvature `c` is optimized end-to-end
2. **Riemannian gradient control** — conformal-factor scaling + Euclidean burn-in
3. **Curriculum training** — progressive data, loss scheduling, hard negatives

The original `usm_big-scale.ipynb` is preserved unchanged for comparison.

---
**Runtime:** GPU recommended (T4 minimum, A100/L4 for full scale)

## 0a. Colab Setup

In [ ]:
import subprocess, sys, os, shutil

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

REPO_URL = 'https://github.com/AdamVanss/the-USM-v2.git'
REPO_DIR = '/content/the-USM-v2'


def find_repo_root():
    """Locate the folder that contains the usm_v2 package."""
    candidates = [
        os.getcwd(),
        os.path.abspath(os.path.join(os.getcwd(), '..')),
        REPO_DIR,
        '/content',
    ]
    for root in candidates:
        if os.path.exists(os.path.join(root, 'usm_v2', '__init__.py')):
            return os.path.abspath(root)
    raise FileNotFoundError(
        'Could not find usm_v2/. Clone the repo or upload a zip (see setup cell).'
    )


def clone_repo(url, dest):
    """Clone repo; reuse existing folder or print a helpful error."""
    pkg = os.path.join(dest, 'usm_v2', '__init__.py')
    if os.path.exists(pkg):
        return

    if os.path.isdir(dest):
        if os.path.isdir(os.path.join(dest, '.git')):
            print(f'Repo already at {dest}, pulling latest...')
            subprocess.check_call(['git', '-C', dest, 'pull', '--ff-only'])
            return
        print(f'Removing incomplete folder: {dest}')
        shutil.rmtree(dest)

    # Private repos: add a Colab secret named GITHUB_TOKEN (GitHub PAT with repo scope)
    clone_url = url
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
            clone_url = url.replace('https://', f'https://{token}@')
            print('Using GITHUB_TOKEN from Colab secrets')
        except Exception:
            print('No GITHUB_TOKEN secret — trying public clone')

    result = subprocess.run(
        ['git', 'clone', '--depth', '1', clone_url, dest],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        err = (result.stderr or result.stdout or '').strip()
        print(err)
        raise RuntimeError(
            'Git clone failed. Common fixes:\n'
            '  1. Private repo → Colab: key icon (Secrets) → add GITHUB_TOKEN (GitHub PAT)\n'
            '  2. Or make the repo public on GitHub\n'
            '  3. Or upload a zip: files.upload() then unzip to /content/the-USM-v2'
        )


if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'geoopt', 'sentence-transformers', 'transformers', 'datasets',
        'umap-learn', 'torchvision'])

    clone_repo(REPO_URL, REPO_DIR)

    ROOT = find_repo_root()
    os.chdir(ROOT)
    if ROOT not in sys.path:
        sys.path.insert(0, ROOT)

    print(f'Working dir: {os.getcwd()}')
    print(f'usm_v2 package: {os.path.exists("usm_v2/__init__.py")}')
else:
    ROOT = find_repo_root()
    if ROOT not in sys.path:
        sys.path.insert(0, ROOT)
    print(f'Local mode — repo root: {ROOT}')

In [ ]:
# Optional: mount Google Drive for persistent checkpoints
CKPT_DIR = '/tmp/usm_v2_checkpoints'

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CKPT_DIR = '/content/drive/MyDrive/usm_v2_checkpoints'
        os.makedirs(CKPT_DIR, exist_ok=True)
        print(f'Checkpoints will be saved to Google Drive: {CKPT_DIR}')
    except Exception as e:
        print(f'Drive mount skipped ({e}), using /tmp')

In [ ]:
import torch
import random
import numpy as np

from usm_v2 import USMConfig, LearnablePoincareBall
from usm_v2.encoders import ConceptEncoder, VisionEncoder
from usm_v2.operators import CompositionalOperator, RelationMaps
from usm_v2.data import (
    load_conceptnet, load_crosslingual, load_snli, load_cifar100,
    CIFAR100_FINE, CIFAR100_COARSE, CIFAR100_FINE2COARSE,
)
from usm_v2.training import train_phase1, train_phase2, _precache_text_embeddings, _precache_clip_embeddings
from usm_v2.evaluation import evaluate_link_prediction, evaluate_crossmodal, evaluate_hierarchy

print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} ({vram:.1f} GB)')
else:
    print('WARNING: No GPU detected — training will be very slow')

## 0b. Configuration

In [ ]:
cfg = USMConfig()
cfg.ckpt_dir = CKPT_DIR

torch.manual_seed(cfg.seed)
random.seed(cfg.seed)
np.random.seed(cfg.seed)

print(f'Dimension:         {cfg.d}')
print(f'Device:            {cfg.device}')
print(f'Large GPU:         {cfg.large_gpu}')
print(f'BF16:              {cfg.use_bf16}')
print(f'Burn-in:           {cfg.burnin_epochs} epochs + {cfg.transition_epochs} transition')
print(f'Curriculum:        {cfg.curriculum_enabled}')
print(f'Learnable c:       {cfg.learnable_curvature}')
print(f'Checkpoints:       {cfg.ckpt_dir}')

## 1. Build Manifold & Models

In [ ]:
manifold = LearnablePoincareBall(
    c_init=cfg.c_init, c_min=cfg.c_min, c_max=cfg.c_max,
    learnable=cfg.learnable_curvature,
).to(cfg.device)

encoder = ConceptEncoder(
    manifold, d_out=cfg.d, backbone=cfg.text_backbone,
    hyperbolic=True, device=cfg.device,
).to(cfg.device)

vis_encoder = VisionEncoder(
    manifold, d_out=cfg.d, clip_model=cfg.clip_model,
    d_clip=cfg.d_clip, hyperbolic=True, device=cfg.device,
).to(cfg.device)

comp_op = CompositionalOperator(manifold, d=cfg.d, hyperbolic=True).to(cfg.device)
rel_maps = RelationMaps(manifold, d=cfg.d, hyperbolic=True).to(cfg.device)

total_params = sum(p.numel() for p in [
    *encoder.proj.parameters(), encoder.mu0,
    *vis_encoder.proj.parameters(), vis_encoder.mu0,
    *comp_op.parameters(), *rel_maps.parameters(),
    manifold._c_param,
])
print(f'Trainable parameters: {total_params:,}')
print(f'Curvature c = {manifold.c.item():.4f}')

## 2. Load Data

In [ ]:
print('--- ConceptNet ---')
triples = load_conceptnet(max_triples=cfg.max_cn_triples)

print('\n--- Cross-lingual ---')
cl_pairs = load_crosslingual(max_per_lang=cfg.max_cl_per_lang)

print('\n--- SNLI ---')
snli_pairs = load_snli(max_pairs=cfg.max_snli_pairs)

vocab_set = set()
for h, _, t in triples:
    vocab_set.add(h)
    vocab_set.add(t)
vocab_set.update(CIFAR100_FINE)
vocab_set.update(CIFAR100_COARSE)
vocab_list = sorted(vocab_set)
concept2bb = {c: i for i, c in enumerate(vocab_list)}

print(f'\nVocabulary: {len(vocab_list):,} concepts')
print(f'Triples:   {len(triples):,}')
print(f'CL pairs:  {len(cl_pairs):,}')
print(f'SNLI:      {len(snli_pairs):,}')

## 3. Pre-cache Embeddings

In [ ]:
vocab_bb = _precache_text_embeddings(encoder, vocab_list, batch_size=cfg.encode_batch, device=cfg.device)
print(f'Vocab backbone cache: {vocab_bb.shape}')

fine_bb = encoder.encode_backbone(CIFAR100_FINE).to(cfg.device)
coarse_bb = encoder.encode_backbone(CIFAR100_COARSE).to(cfg.device)
fine2coarse_tensor = torch.tensor(
    [CIFAR100_FINE2COARSE[i] for i in range(len(CIFAR100_FINE))],
    device=cfg.device,
)
print(f'Fine BB: {fine_bb.shape}, Coarse BB: {coarse_bb.shape}')

random.shuffle(triples)
n_test = max(500, len(triples) // 20)
test_triples = triples[:n_test]
train_triples = triples[n_test:]
print(f'Train triples: {len(train_triples):,}  |  Test triples: {len(test_triples):,}')

## 4. Phase 1 — Text-only Training

Integrates burn-in (Euclidean -> hyperbolic), curriculum sampling, and learnable curvature.

In [ ]:
history_p1 = train_phase1(
    cfg, manifold, encoder, comp_op, rel_maps,
    train_triples, cl_pairs, snli_pairs,
    vocab_list, vocab_bb, concept2bb,
)
print(f'\nFinal curvature c = {manifold.c.item():.4f}')

## 5. Phase 1 Evaluation

In [ ]:
lp_results = evaluate_link_prediction(
    encoder, rel_maps, test_triples, vocab_list,
    vocab_bb, concept2bb, manifold.c,
    hyperbolic=True, tag='P1', device=cfg.device,
)
print('Link Prediction (Phase 1):')
for k, v in lp_results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

## 6. Phase 2 — Multimodal Training

Aligns CLIP vision embeddings with the learned text geometry + hierarchy objectives.

In [ ]:
from torch.utils.data import DataLoader

print('Loading CIFAR-100...')
cifar_train, cifar_test = load_cifar100(data_root='./data')

cifar_dl_train = DataLoader(cifar_train, batch_size=256, shuffle=False, num_workers=2)
cifar_dl_test = DataLoader(cifar_test, batch_size=256, shuffle=False, num_workers=2)

print('Pre-caching CLIP embeddings...')
clip_train, clip_labels_train = _precache_clip_embeddings(vis_encoder, cifar_dl_train, device=cfg.device)
clip_test, clip_labels_test = _precache_clip_embeddings(vis_encoder, cifar_dl_test, device=cfg.device)
print(f'Train CLIP: {clip_train.shape}, Test CLIP: {clip_test.shape}')

In [ ]:
history_p2 = train_phase2(
    cfg, manifold, encoder, vis_encoder,
    clip_train, clip_labels_train,
    fine_bb, coarse_bb, fine2coarse_tensor,
)
print(f'\nFinal curvature c = {manifold.c.item():.4f}')

## 7. Phase 2 Evaluation

In [ ]:
fine_z = encoder.project(fine_bb, c=manifold.c)

xm_results = evaluate_crossmodal(
    vis_encoder, fine_z, clip_test, clip_labels_test,
    manifold.c, hyperbolic=True, tag='P2', device=cfg.device,
)
print('Cross-modal Retrieval (Phase 2):')
for k, v in xm_results.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')

hier_results = evaluate_hierarchy(
    vis_encoder, encoder, fine_bb, coarse_bb,
    fine2coarse_tensor, manifold.c, hyperbolic=True, tag='P2',
)
print(f'\nHierarchy Accuracy: {hier_results["accuracy"]:.2%}')
print(f'Mean depth diff (fine - coarse): {hier_results["mean_diff"]:.4f}')

## 8. Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].plot(history_p1['loss_total'], label='Total')
axes[0, 0].plot(history_p1['loss_rel'], label='L_rel', alpha=0.7)
axes[0, 0].plot(history_p1['loss_cl'], label='L_cl', alpha=0.7)
axes[0, 0].set_title('Phase 1 Losses')
axes[0, 0].legend()
axes[0, 0].set_xlabel('Epoch')

axes[0, 1].plot(history_p1['curvature'], label='Phase 1', color='tab:blue')
if history_p2['curvature']:
    offset = len(history_p1['curvature'])
    axes[0, 1].plot(range(offset, offset + len(history_p2['curvature'])),
                    history_p2['curvature'], label='Phase 2', color='tab:orange')
axes[0, 1].set_title('Learned Curvature c')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5, label='c=1 (v1 fixed)')
axes[0, 1].legend()

axes[0, 2].plot(history_p1['curriculum_pct'], color='tab:green')
axes[0, 2].set_title('Curriculum Progress')
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylabel('Data fraction')

if history_p2['loss_total']:
    axes[1, 0].plot(history_p2['loss_total'])
    axes[1, 0].set_title('Phase 2 Loss')
    axes[1, 0].set_xlabel('Epoch')

axes[1, 1].hist(hier_results['fine_depths'], bins=30, alpha=0.6, label='Fine (specific)')
axes[1, 1].hist(hier_results['coarse_depths'], bins=30, alpha=0.6, label='Coarse (general)')
axes[1, 1].set_title(f'Hierarchy Depth (acc={hier_results["accuracy"]:.0%})')
axes[1, 1].legend()
axes[1, 1].set_xlabel('dist(z, origin)')

try:
    import umap
    from usm_v2.manifold import logmap0
    with torch.no_grad():
        sample_idx = torch.randperm(len(vocab_list))[:500]
        sample_bb = vocab_bb[sample_idx]
        z_sample = encoder.project(sample_bb, c=manifold.c)
        tangent = logmap0(z_sample, manifold.c).cpu().numpy()
    reducer = umap.UMAP(n_components=2, random_state=42)
    umap_2d = reducer.fit_transform(tangent)
    norms = np.linalg.norm(tangent, axis=1)
    sc = axes[1, 2].scatter(umap_2d[:, 0], umap_2d[:, 1], c=norms, cmap='viridis', s=8, alpha=0.7)
    plt.colorbar(sc, ax=axes[1, 2], label='Tangent norm (depth)')
    axes[1, 2].set_title('UMAP of Concepts (color=depth)')
except ImportError:
    axes[1, 2].text(0.5, 0.5, 'pip install umap-learn\nfor UMAP visualization',
                    ha='center', va='center', transform=axes[1, 2].transAxes)

plt.tight_layout()
plt.savefig('usm_v2_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: usm_v2_results.png')

## 9. Save Final Model to Drive

In [ ]:
final_path = os.path.join(CKPT_DIR, 'usm_v2_final.pt')
os.makedirs(CKPT_DIR, exist_ok=True)

torch.save({
    'manifold': manifold.state_dict(),
    'encoder': encoder.state_dict(),
    'vis_encoder': vis_encoder.state_dict(),
    'comp_op': comp_op.state_dict(),
    'rel_maps': rel_maps.state_dict(),
    'config': cfg,
    'history_p1': history_p1,
    'history_p2': history_p2,
    'lp_results': lp_results,
    'xm_results': xm_results,
    'hier_results': hier_results,
    'final_curvature': manifold.c.item(),
}, final_path)

print(f'Saved final model + results to {final_path}')
print(f'Final curvature c = {manifold.c.item():.4f}')

## 10. Summary

| Metric | v1 (fixed c=1) | v2 (learned c) |
|--------|----------------|----------------|
| MRR | ... | ... |
| Hits@10 | ... | ... |
| Hierarchy Acc | ... | ... |
| Cross-modal R@5 | ... | ... |
| Final curvature | 1.0 | ... |

Fill in after running both notebooks.